# Exploring the Dataset: Building the System CPU Metrics Table

**Goal:** Understand the structure of the `system.cpu` metricbeat logs, identify key fields, and map them to the `system_cpu_events` database table.

**Source file:** `gather/internal_share/logs/2022-01-21-system_cpu.log`  
**What it contains:** CPU utilisation snapshots from the internal-share server, collected every 45 seconds via metricbeat throughout 2022-01-21.

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below — everything else derives from it.

In [1]:
from pathlib import Path

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path(r"C:\Users\ishaanshetty\DATA-201\russellmitchell")

cpu_log_path = DATASET_ROOT / "gather" / "internal_share" / "logs" / "2022-01-21-system_cpu.log"

print(f"Dataset found at: {DATASET_ROOT}")

Dataset found at: C:\Users\ishaanshetty\DATA-201\russellmitchell


## 1. Load the Raw CPU Log File

The file is a JSON Lines document — each line is one metricbeat CPU snapshot with deeply nested fields. We flatten it with `json_normalize` and rename the key fields to clean column names.

In [2]:
import json

import pandas as pd

raw_records = []
with open(cpu_log_path) as f:
    for line in f:
        raw_records.append(json.loads(line.strip()))

df_raw = pd.json_normalize(raw_records)

# Extract and rename the columns we care about
df = df_raw[
    [
        "@timestamp",
        "host.name",
        "system.cpu.total.pct",
        "system.cpu.user.pct",
        "system.cpu.system.pct",
        "system.cpu.idle.pct",
        "system.cpu.iowait.pct",
        "system.cpu.steal.pct",
        "system.cpu.softirq.pct",
        "system.cpu.cores",
        "event.duration",
        "metricset.period",
    ]
].copy()

df.columns = [
    "timestamp",
    "host",
    "cpu_total_pct",
    "cpu_user_pct",
    "cpu_system_pct",
    "cpu_idle_pct",
    "cpu_iowait_pct",
    "cpu_steal_pct",
    "cpu_softirq_pct",
    "cores",
    "event_duration_ns",
    "metricset_period_ms",
]

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded {len(df)} records")
print(f"\nColumns: {list(df.columns)}")
print(f"\nHost: {df['host'].iloc[0]}")
print(f"Date: {df['timestamp'].dt.date.iloc[0]}")
print(
    f"Time range: {df['timestamp'].min().strftime('%H:%M:%S')} UTC to {df['timestamp'].max().strftime('%H:%M:%S')} UTC"
)
print(
    f"Sampling interval: {df['metricset_period_ms'].iloc[0] / 1000:.0f}s  |  CPU cores: {int(df['cores'].iloc[0])}"
)

Loaded 1920 records

Columns: ['timestamp', 'host', 'cpu_total_pct', 'cpu_user_pct', 'cpu_system_pct', 'cpu_idle_pct', 'cpu_iowait_pct', 'cpu_steal_pct', 'cpu_softirq_pct', 'cores', 'event_duration_ns', 'metricset_period_ms']

Host: internal-share
Date: 2022-01-21
Time range: 00:00:22 UTC to 23:59:37 UTC
Sampling interval: 45s  |  CPU cores: 1


## 2. Examine a Single Record

Before flattening, each record is a deeply nested JSON object from metricbeat. Here is what the raw structure looks like.

In [3]:
import json

print("Raw data for record 0:\n")
print(json.dumps(raw_records[0], indent=2))

Raw data for record 0:

{
  "service": {
    "type": "system"
  },
  "agent": {
    "hostname": "internal-share",
    "name": "internal-share",
    "id": "480761c0-a9a7-48cc-a30b-f67100b44955",
    "ephemeral_id": "cc234e3a-114f-402c-ab60-737cee803231",
    "version": "7.13.2",
    "type": "metricbeat"
  },
  "event": {
    "module": "system",
    "dataset": "system.cpu",
    "duration": 7100828
  },
  "@version": "1",
  "metricset": {
    "period": 45000,
    "name": "cpu"
  },
  "host": {
    "cpu": {
      "pct": 0.0561
    },
    "name": "internal-share"
  },
  "ecs": {
    "version": "1.9.0"
  },
  "@timestamp": "2022-01-21T00:00:22.531Z",
  "tags": [
    "beats_input_raw_event"
  ],
  "system": {
    "cpu": {
      "iowait": {
        "norm": {
          "pct": 0.0012
        },
        "pct": 0.0012
      },
      "steal": {
        "pct": 0.0007,
        "norm": {
          "pct": 0.0007
        }
      },
      "irq": {
        "pct": 0,
        "norm": {
          "pct": 0
  

### What do these fields mean?

| Field | What It Is | Example |
|-------|------------|--------|
| `@timestamp` | When metricbeat collected this sample | `2022-01-21T00:00:22.531Z` |
| `system.cpu.total.pct` | Total CPU usage across all states | `0.0561` (5.6%) |
| `system.cpu.user.pct` | Time running user-space processes | `0.0273` |
| `system.cpu.system.pct` | Time in kernel / system calls | `0.0271` |
| `system.cpu.idle.pct` | Time the CPU was idle | `0.9427` |
| `system.cpu.iowait.pct` | Time waiting for I/O to complete | `0.0012` |
| `system.cpu.steal.pct` | Time stolen by the hypervisor | `0.0007` |
| `event.duration` | How long metricbeat took to collect (ns) | `7100828` |
| `metricset.period` | Configured collection interval (ms) | `45000` |

### 2.1 Record structure

Each line in the CPU log is a JSON object produced by metricbeat. The raw structure is deeply nested:
```json
{
  "@timestamp": "2022-01-21T00:00:22.531Z",
  "host": {"name": "internal-share"},
  "system": {
    "cpu": {
      "total": {"pct": 0.0561},
      "user":  {"pct": 0.0273},
      ...
    }
  },
  "event": {"duration": 7100828},
  "metricset": {"period": 45000}
}
```
`pd.json_normalize` flattens this into dotted column names (e.g. `system.cpu.total.pct`), which are then renamed to clean column names.


### 2.2 Parsing strategy

| Raw dotted field | Renamed column | Notes |
|-----------------|----------------|-------|
| `@timestamp` | `timestamp` | Parsed to `datetime64[ns, UTC]` |
| `host.name` | `host` | Server identifier string |
| `system.cpu.total.pct` | `cpu_total_pct` | Sum of all CPU states |
| `system.cpu.user.pct` | `cpu_user_pct` | User-space processes |
| `system.cpu.system.pct` | `cpu_system_pct` | Kernel/system calls |
| `system.cpu.idle.pct` | `cpu_idle_pct` | Idle time |
| `system.cpu.iowait.pct` | `cpu_iowait_pct` | Waiting for I/O |
| `system.cpu.steal.pct` | `cpu_steal_pct` | Hypervisor steal |
| `system.cpu.softirq.pct` | `cpu_softirq_pct` | Software interrupts |
| `system.cpu.cores` | `cores` | Logical core count |
| `event.duration` | `event_duration_ns` | Collection time in nanoseconds |
| `metricset.period` | `metricset_period_ms` | Configured interval in milliseconds |


## 3. Field-by-Field Exploration

### 3.1 timestamp


In [ ]:
print("=== Timestamp range ===")
print(f"  Earliest: {df['timestamp'].min()}")
print(f"  Latest:   {df['timestamp'].max()}")
print(f"  Span:     {df['timestamp'].max() - df['timestamp'].min()}")
print(f"  Records:  {len(df)}")
print(
    f"  Expected at 45s interval: {int((df['timestamp'].max() - df['timestamp'].min()).total_seconds() / 45) + 1}"
)
print()
dates = df["timestamp"].dt.date
for date, count in dates.value_counts().sort_index().items():
    print(f"  {date}: {count} records")

### 3.2 host


In [ ]:
print("=== host distribution ===")
for host, count in df["host"].value_counts().items():
    print(f"  {host}: {count}")

### 3.3 cpu pct fields


In [ ]:
pct_cols = [
    "cpu_total_pct",
    "cpu_user_pct",
    "cpu_system_pct",
    "cpu_idle_pct",
    "cpu_iowait_pct",
    "cpu_steal_pct",
    "cpu_softirq_pct",
]
print("=== CPU pct field ranges ===")
for col in pct_cols:
    print(
        f"  {col:20s}  min={df[col].min():.4f}  max={df[col].max():.4f}  mean={df[col].mean():.4f}  null={df[col].isna().sum()}"
    )

### 3.4 cores, event_duration_ns, metricset_period_ms


In [ ]:
print("=== cores ===")
print(df["cores"].value_counts().to_string())
print()
print("=== event_duration_ns ===")
print(df["event_duration_ns"].describe())
print()
print("=== metricset_period_ms ===")
print(df["metricset_period_ms"].value_counts().to_string())

## 4. Raw 1:1 DataFrame

The full unnormalized `df_raw` produced by `json_normalize` — one row per metricbeat snapshot, all original dotted field names preserved before renaming. This is the direct source for the `system_cpu_events` table.


In [ ]:
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(3)

## 5. DataFrame Schema

The cleaned `df` with renamed columns and their types.


In [3]:
schema_rows = [
    ("timestamp", "datetime64[ns, UTC]", "Metricbeat collection timestamp"),
    ("host", "str", "Hostname of the monitored server"),
    ("cpu_total_pct", "float64", "Total CPU usage (all states combined)"),
    ("cpu_user_pct", "float64", "CPU time in user-space processes"),
    ("cpu_system_pct", "float64", "CPU time in kernel/system calls"),
    ("cpu_idle_pct", "float64", "CPU idle time"),
    ("cpu_iowait_pct", "float64", "CPU waiting on I/O"),
    ("cpu_steal_pct", "float64", "CPU stolen by hypervisor"),
    ("cpu_softirq_pct", "float64", "CPU handling software interrupts"),
    ("cores", "int64", "Number of logical CPU cores"),
    ("event_duration_ns", "int64", "Time metricbeat took to collect (ns)"),
    ("metricset_period_ms", "int64", "Configured collection interval (ms)"),
]
pd.DataFrame(schema_rows, columns=["field", "type", "description"])

,field,type,description
0,timestamp,"datetime64[ns, UTC]",Metricbeat collection timestamp
1,host,str,Hostname of the monitored server
2,cpu_total_pct,float64,Total CPU usage (all states combined)
3,cpu_user_pct,float64,CPU time in user-space processes
4,cpu_system_pct,float64,CPU time in kernel/system calls
5,cpu_idle_pct,float64,CPU idle time
6,cpu_iowait_pct,float64,CPU waiting on I/O
7,cpu_steal_pct,float64,CPU stolen by hypervisor
8,cpu_softirq_pct,float64,CPU handling software interrupts
9,cores,int64,Number of logical CPU cores


## 6. Summary Statistics

Descriptive stats across the full day. The mean and median (50%) for `cpu_total_pct` are very close (~6.9–7.7%), indicating a stable baseline for most of the day. The max of 100% and the high `cpu_iowait_pct` max of 94.4% reveal two distinct attack windows where the server was saturated.

In [4]:
df[
    ["cpu_total_pct", "cpu_user_pct", "cpu_system_pct", "cpu_idle_pct", "cpu_iowait_pct"]
].describe().round(4)

,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_idle_pct,cpu_iowait_pct
stat,,,,,
count,1920.0000,1920.0000,1920.0000,1920.0000,1920.0000
mean,0.0767,0.0327,0.0377,0.9207,0.0027
std,0.0869,0.0400,0.0150,0.0950,0.0282
min,0.0359,0.0156,0.0139,0.0000,0.0000
25%,0.0659,0.0284,0.0351,0.9266,0.0006
50%,0.0691,0.0299,0.0373,0.9297,0.0010
75%,0.0721,0.0313,0.0395,0.9329,0.0017
max,1.0000,0.7303,0.3116,0.9625,0.9437


## 7. Percentile Distribution

The p99 jumps dramatically to 38.3%, far above the p95 of 7.9%, which shows the spike records are extreme outliers. 99% of the day sits below 8% total CPU — the remaining 1% (19 records) accounts for the two attack windows.

In [5]:
pcts = [50, 75, 90, 95, 99, 100]
cols = ["cpu_total_pct", "cpu_user_pct", "cpu_system_pct", "cpu_iowait_pct"]
rows = []
for p in pcts:
    row = {"percentile": f"p{p}"}
    for col in cols:
        row[col] = round(df[col].quantile(p / 100), 4)
    rows.append(row)
pd.DataFrame(rows)

,percentile,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_iowait_pct
0,p50,0.0691,0.0299,0.0373,0.0010
1,p75,0.0721,0.0313,0.0395,0.0017
2,p90,0.0768,0.0326,0.0425,0.0024
3,p95,0.0794,0.0333,0.0448,0.0033
4,p99,0.3826,0.0376,0.0827,0.0077
5,p100,1.0000,0.7303,0.3116,0.9437


## 8. CPU Spike Records

Records where `cpu_total_pct > 0.5` (50%). There are **19 spike records** across **two distinct windows**:

- **Window 1 — 00:15–00:18 UTC** (5 records): `cpu_total_pct` pegged at 100% with very low `cpu_user_pct`, suggesting a kernel/system-level saturation. `cpu_idle_pct` drops to 0.000.
- **Window 2 — 06:13–06:22 UTC** (14 records): A longer, sustained burst with elevated `cpu_user_pct` (up to 73%) and `cpu_iowait_pct` (up to 94.4%), consistent with intensive user-space computation combined with heavy I/O — the signature of active data exfiltration or file staging.

In [6]:
df_spike = df[df["cpu_total_pct"] > 0.5][
    ["timestamp", "cpu_total_pct", "cpu_user_pct", "cpu_system_pct", "cpu_iowait_pct"]
].reset_index(drop=True)
df_spike

,timestamp,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_iowait_pct
0,2022-01-21 00:15:22.531000+00:00,1.0000,0.0196,0.1076,0.0000
1,2022-01-21 00:16:07.531000+00:00,1.0000,0.0167,0.1029,0.0000
2,2022-01-21 00:16:52.531000+00:00,1.0000,0.0156,0.0998,0.0000
3,2022-01-21 00:17:37.531000+00:00,1.0000,0.0485,0.1258,0.0000
4,2022-01-21 00:18:22.531000+00:00,0.5854,0.4469,0.1285,0.0014
5,2022-01-21 06:13:07.531000+00:00,0.9885,0.0467,0.1205,0.0113
6,2022-01-21 06:13:52.531000+00:00,1.0000,0.0198,0.1071,0.0000
7,2022-01-21 06:14:37.531000+00:00,1.0000,0.0169,0.1100,0.0000
8,2022-01-21 06:15:22.531000+00:00,1.0000,0.0173,0.1074,0.0000
9,2022-01-21 06:16:07.531000+00:00,0.9944,0.7303,0.1325,0.0056


## 9. Spike Windows in Context

### 9.1 Window 1 — 00:15 UTC

Three baseline samples before and after the first spike cluster. Shows a sharp ramp from ~7% to 100% starting at 00:14, then a rapid drop back to baseline by 00:19. `cpu_user_pct` during the peak is very low (~2–5%), meaning the saturation is kernel-driven — consistent with a process monopolising the scheduler or triggering a tight kernel loop.

In [7]:
# Window 1: first spike at ~00:15 UTC
w1_first = df[df["cpu_total_pct"] > 0.5].index[0]
w1_last = df[(df["cpu_total_pct"] > 0.5) & (df["timestamp"] < "2022-01-21 01:00:00+00:00")].index[
    -1
]
ctx1 = df.iloc[max(0, w1_first - 3) : w1_last + 4][
    ["timestamp", "cpu_total_pct", "cpu_user_pct", "cpu_system_pct", "cpu_iowait_pct"]
].reset_index(drop=True)
ctx1

,timestamp,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_iowait_pct
0,2022-01-21 00:13:07.531000+00:00,0.0670,0.0298,0.0353,0.0009
1,2022-01-21 00:13:52.531000+00:00,0.0694,0.0302,0.0373,0.0014
2,2022-01-21 00:14:37.531000+00:00,0.4223,0.1677,0.0847,0.0053
3,2022-01-21 00:15:22.531000+00:00,1.0000,0.0196,0.1076,0.0000
4,2022-01-21 00:16:07.531000+00:00,1.0000,0.0167,0.1029,0.0000
5,2022-01-21 00:16:52.531000+00:00,1.0000,0.0156,0.0998,0.0000
6,2022-01-21 00:17:37.531000+00:00,1.0000,0.0485,0.1258,0.0000
7,2022-01-21 00:18:22.531000+00:00,0.5854,0.4469,0.1285,0.0014
8,2022-01-21 00:19:07.531000+00:00,0.1527,0.0901,0.0581,0.0109
9,2022-01-21 00:19:52.531000+00:00,0.0715,0.0302,0.0392,0.0005


### 9.2 Window 2 — 06:13 UTC

The second and larger spike starting at 06:13 UTC. The first three samples show the same 100% kernel-saturation pattern as Window 1, then transition into high `cpu_user_pct` (up to 73%) and very high `cpu_iowait_pct` (up to 94.4%) — the CPU is simultaneously doing heavy computation and waiting on disk I/O. This is consistent with file compression, encryption, or bulk data transfer.

In [8]:
# Window 2: second spike at ~06:13 UTC
w2_first = df[(df["cpu_total_pct"] > 0.5) & (df["timestamp"] > "2022-01-21 06:00:00+00:00")].index[
    0
]
w2_last = df[(df["cpu_total_pct"] > 0.5) & (df["timestamp"] > "2022-01-21 06:00:00+00:00")].index[
    -1
]
ctx2 = df.iloc[max(0, w2_first - 3) : w2_last + 4][
    ["timestamp", "cpu_total_pct", "cpu_user_pct", "cpu_system_pct", "cpu_iowait_pct"]
].reset_index(drop=True)
ctx2

,timestamp,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_iowait_pct
0,2022-01-21 06:10:52.531000+00:00,0.0721,0.0307,0.0395,0.0007
1,2022-01-21 06:11:37.531000+00:00,0.0782,0.0311,0.0447,0.0016
2,2022-01-21 06:12:22.531000+00:00,0.0703,0.0293,0.0392,0.0005
3,2022-01-21 06:13:07.531000+00:00,0.9885,0.0467,0.1205,0.0113
4,2022-01-21 06:13:52.531000+00:00,1.0000,0.0198,0.1071,0.0000
5,2022-01-21 06:14:37.531000+00:00,1.0000,0.0169,0.1100,0.0000
6,2022-01-21 06:15:22.531000+00:00,1.0000,0.0173,0.1074,0.0000
7,2022-01-21 06:16:07.531000+00:00,0.9944,0.7303,0.1325,0.0056
8,2022-01-21 06:16:52.531000+00:00,0.8946,0.6953,0.1518,0.1036
9,2022-01-21 06:17:37.531000+00:00,0.9575,0.5169,0.2340,0.0354


## 10. Database Table Preview

### 10.1 Table Preview

The first 10 rows as they will appear in the `system_cpu_events` PostgreSQL table, with a surrogate integer primary key.

In [9]:
# Preview of the system_cpu_events table as it will look in PostgreSQL
db_preview = (
    df[
        [
            "timestamp",
            "cpu_total_pct",
            "cpu_user_pct",
            "cpu_system_pct",
            "cpu_idle_pct",
            "cpu_iowait_pct",
        ]
    ]
    .head(10)
    .reset_index(drop=True)
)
db_preview.insert(0, "system_cpu_id", range(1, 11))
db_preview

,system_cpu_id,timestamp,cpu_total_pct,cpu_user_pct,cpu_system_pct,cpu_idle_pct,cpu_iowait_pct
0,1,2022-01-21 00:00:22.531000+00:00,0.0561,0.0273,0.0271,0.9427,0.0012
1,2,2022-01-21 00:01:07.531000+00:00,0.0671,0.0288,0.0373,0.9317,0.0012
2,3,2022-01-21 00:01:52.531000+00:00,0.0644,0.0286,0.0346,0.9332,0.0024
3,4,2022-01-21 00:02:37.531000+00:00,0.0668,0.0276,0.0385,0.9320,0.0012
4,5,2022-01-21 00:03:22.531000+00:00,0.0655,0.0277,0.0364,0.9340,0.0005
5,6,2022-01-21 00:04:07.531000+00:00,0.0691,0.0311,0.0361,0.9297,0.0012
6,7,2022-01-21 00:04:52.531000+00:00,0.0715,0.0302,0.0391,0.9276,0.0009
7,8,2022-01-21 00:05:37.531000+00:00,0.0700,0.0312,0.0364,0.9288,0.0012
8,9,2022-01-21 00:06:22.531000+00:00,0.0737,0.0308,0.0406,0.9244,0.0019
9,10,2022-01-21 00:07:07.531000+00:00,0.0729,0.0307,0.0399,0.9257,0.0014


In [ ]:
postgresql_ddl = """
-- PostgreSQL
CREATE TABLE system_cpu_events (
    system_cpu_event_id SERIAL PRIMARY KEY,
    event_timestamp     TIMESTAMP WITH TIME ZONE NOT NULL,
    hostname            TEXT,
    cpu_total_pct       NUMERIC(6,4),
    cpu_user_pct        NUMERIC(6,4),
    cpu_system_pct      NUMERIC(6,4),
    cpu_idle_pct        NUMERIC(6,4),
    cpu_iowait_pct      NUMERIC(6,4),
    cpu_steal_pct       NUMERIC(6,4),
    cpu_softirq_pct     NUMERIC(6,4),
    cpu_cores           SMALLINT,
    created_at          TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

mysql_ddl = """
-- MySQL
CREATE TABLE system_cpu_events (
    system_cpu_event_id INT AUTO_INCREMENT PRIMARY KEY,
    event_timestamp     DATETIME NOT NULL,
    hostname            VARCHAR(255),
    cpu_total_pct       DECIMAL(6,4),
    cpu_user_pct        DECIMAL(6,4),
    cpu_system_pct      DECIMAL(6,4),
    cpu_idle_pct        DECIMAL(6,4),
    cpu_iowait_pct      DECIMAL(6,4),
    cpu_steal_pct       DECIMAL(6,4),
    cpu_softirq_pct     DECIMAL(6,4),
    cpu_cores           SMALLINT,
    created_at          DATETIME DEFAULT CURRENT_TIMESTAMP
);
"""

print(postgresql_ddl)
print(mysql_ddl)

### 10.2 Raw DDL


In [ ]:
print("=== Summary ===")
print(f"Total records:        {len(df)}")
print(f"Time range:           {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Span:                 {df['timestamp'].max() - df['timestamp'].min()}")
print(f"Distinct hosts:       {df['host'].nunique()}")
print(f"CPU cores (observed): {df['cores'].unique()}")
print(f"Columns in df:        {len(df.columns)}")
print()

print("Columns with nulls:")
for col in df.columns:
    n = df[col].isna().sum()
    if n > 0:
        print(f"  {col}: {n} nulls ({n / len(df) * 100:.1f}%)")
print()

spike_count = (df["cpu_total_pct"] > 0.5).sum()
print(f"Spike records (>50% CPU): {spike_count} ({spike_count / len(df) * 100:.1f}%)")
print(
    f"Normal records (<50% CPU): {len(df) - spike_count} ({(len(df) - spike_count) / len(df) * 100:.1f}%)"
)

## 11. Normalization Observations

Applying the `normalization_rules_sheet.md` checklist to the raw system CPU metrics data.

### 11.1 1NF Check

**Atomic fields:** All columns after flattening with `json_normalize` are scalar values (timestamps, floats, integers, strings). No arrays or nested structures remain in the working DataFrame.

**Repeating groups:** None. The CPU percentage fields (`cpu_total_pct`, `cpu_user_pct`, `cpu_system_pct`, etc.) are separate named columns, not a repeating group — each measures a distinct CPU state.

**1NF status: satisfied.** All fields are atomic after flattening.

### 11.2 2NF Check

**Primary key:** `system_cpu_event_id` is a single-column surrogate PK. Partial dependencies require a composite key, which does not exist here.

**2NF status: satisfied.** Single-column PK makes partial dependencies impossible.

### 11.3 3NF Check

**Transitive dependencies identified:**

| Determinant | Dependent(s) | Notes |
|-------------|-------------|-------|
| `hostname` | server configuration | The hostname could determine `cpu_cores` if cores are fixed per server. In this dataset there is only one host, so this cannot be verified. |
| `cpu_total_pct` | derived sum | `cpu_total_pct` is approximately the sum of `cpu_user_pct + cpu_system_pct + cpu_iowait_pct + cpu_steal_pct + cpu_softirq_pct`. It is a computed aggregate, not independently measured. |

**3NF status: acceptable.** `cpu_total_pct` is technically derivable from the other pct columns but is stored for query convenience. The `hostname -> cpu_cores` dependency is deferred — if multiple hosts are added, `cpu_cores` should move to a hosts dimension table.

### 11.4 Preliminary Functional Dependencies

| FD | Determinant | Dependent(s) | Reasoning |
|---|---|---|---|
| FD1 | system_cpu_event_id | all attributes | Surrogate PK, trivially determines everything. |
| FD2 | event_timestamp | all metrics | Timestamps are unique per collection interval (45s). Candidate natural key. |
| FD3 | hostname | cpu_cores | If cores are fixed per host, hostname functionally determines cpu_cores. Relevant when multiple hosts are added. |
| FD4 | cpu_user_pct, cpu_system_pct, cpu_iowait_pct, cpu_steal_pct, cpu_softirq_pct | cpu_total_pct | Total is the sum of components. Stored denormalized for convenience. |

## 12. Key Findings for Schema Design

1. **Clean, flat structure:** After `json_normalize`, all metricbeat fields are scalar. No 1NF violations — the schema is identical in shape to the intranet-server log explored previously.

2. **`cpu_total_pct` is a derived column:** It is approximately the sum of the other pct fields. Stored for query convenience but could be dropped in a strict normalized schema and computed as a view.

3. **Two distinct attack windows:** Unlike the single spike window in the intranet-server log, this file contains two bursts — one at **00:15 UTC** (5 records, ~3 minutes, kernel-saturated) and one at **06:13 UTC** (14 records, ~10 minutes, user+I/O heavy). The 06:13 window has the highest `cpu_iowait_pct` values seen across all logs (up to 94.4%), pointing to bulk file read/write activity during that period.

4. **Same host as the 2022-01-22 log:** Both `2022-01-21` and `2022-01-22` originate from `internal-share`. The 2022-01-22 log shows no equivalent spike, confirming the attack activity was confined to 2022-01-21.

5. **Collection interval is fixed:** `metricset_period_ms` is always 45,000 ms. This column adds no analytical value in a single-source table and could be dropped or moved to a metadata table.

6. **No label integration:** CPU records have no direct annotation labels. Attack detection relies on threshold comparison (`cpu_total_pct > 0.5`) cross-referenced with timestamps from the access log.

7. **NUMERIC(6,4) is appropriate:** All pct values range 0.0–1.0 with 4 decimal places of precision in the source data. `NUMERIC(6,4)` covers the full range without floating-point rounding errors. Note: `cpu_total_pct` reaches exactly 1.0000 during the attack, so the type must accommodate values up to 1.0.
